## API endpoint data ingestion

Load endpoint from OpenAPI specification YAML or JSON file. Parse the specification and extract the following data for the next pipeline:
- path
- method
- summary
- description
- operation id
- parameters
- request body if method is POST, PUT, PATCH
- response schema for 200 status codes

In [30]:
import yaml
from pathlib import Path
import os
import json
from pydantic import BaseModel, Field
from typing import Any, Dict, List, Optional
from dotenv import load_dotenv
from agno.agent import Agent
from agno.models.google import Gemini
from agno.utils.pprint import pprint_run_response

load_dotenv()

True

### load openapi spec

yaml or json file

In [15]:
spec_path: str = os.path.abspath("data/route.yaml")
spec = None
with open(spec_path, 'r') as f:
  spec = yaml.safe_load(f)

# spec
print(spec['paths']['/users']['get']['parameters'])
print(spec['paths']['/users']['get']['responses'])

[{'name': 'page', 'in': 'query', 'description': 'Page number (1-based)', 'required': False, 'schema': {'type': 'integer', 'minimum': 1, 'default': 1}}, {'name': 'limit', 'in': 'query', 'description': 'Number of users per page', 'required': False, 'schema': {'type': 'integer', 'minimum': 1, 'maximum': 100, 'default': 20}}, {'name': 'role', 'in': 'query', 'description': 'Filter users by role', 'required': False, 'schema': {'type': 'string', 'enum': ['admin', 'user', 'guest']}}]
{'200': {'description': 'Successful response with list of users', 'content': {'application/json': {'schema': {'type': 'object', 'properties': {'users': {'type': 'array', 'items': {'$ref': '#/components/schemas/User'}}, 'total': {'type': 'integer', 'description': 'Total number of users matching filter'}, 'page': {'type': 'integer'}, 'limit': {'type': 'integer'}}}}}}, '400': {'description': 'Invalid query parameters'}, '500': {'description': 'Internal server error'}}


### extract endpoint information

path, method, summary, description, operationid, parameters, responses

In [16]:
class EndpointInfo(BaseModel):
  path: str
  method: str
  summary: Optional[str] = None
  description: Optional[str] = None
  operationId: Optional[str] = None
  parameters: List[Any] = Field(default_factory=list)
  responses: Dict[str, Any] = Field(default_factory=dict)

method = "get"
operation = spec['paths']['/users']['get']
parameters = operation.get('parameters', [])
responses = operation.get('responses', {})
endpoint = EndpointInfo(
  path="/users",
  method=method,
  summary=operation.get('summary'),
  description=operation.get('description'),
  operationId=operation.get('operationId'),
  parameters=parameters,
  responses=responses,
)
endpoint

EndpointInfo(path='/users', method='get', summary='List users', description='Retrieves a paginated list of users with optional filtering by role.', operationId='listUsers', parameters=[{'name': 'page', 'in': 'query', 'description': 'Page number (1-based)', 'required': False, 'schema': {'type': 'integer', 'minimum': 1, 'default': 1}}, {'name': 'limit', 'in': 'query', 'description': 'Number of users per page', 'required': False, 'schema': {'type': 'integer', 'minimum': 1, 'maximum': 100, 'default': 20}}, {'name': 'role', 'in': 'query', 'description': 'Filter users by role', 'required': False, 'schema': {'type': 'string', 'enum': ['admin', 'user', 'guest']}}], responses={'200': {'description': 'Successful response with list of users', 'content': {'application/json': {'schema': {'type': 'object', 'properties': {'users': {'type': 'array', 'items': {'$ref': '#/components/schemas/User'}}, 'total': {'type': 'integer', 'description': 'Total number of users matching filter'}, 'page': {'type': 'i

### instantiate llm agent

using Agno instantiate llm agent

In [ ]:
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

agent = Agent(
  model=Gemini(id="gemini-2.5-flash-lite"),
	description="Endpoint analyzer",
  instructions="""
	You are an expert API endpoint analyst. Given an OpenAPI endpoint specification as JSON, generate:
	1. High-level goal and use case.
	2. Explanation of the API endpoint with detail request-response cycle and working flow.
	Be concise, structured, and accurate. Output in Markdown.
""",
markdown=True
)

agent

### run the agent with the prompt and openapi json payload

In [ ]:
spec_data = endpoint.model_dump(exclude_none=True)
prompt = f"""Analyze this API endpoint: 
<json>
{json.dumps(spec_data)}
</json>
"""
agent_res = agent.run(prompt)
agent_res

In [31]:
pprint_run_response(agent_res, markdown=True)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃                                     API Endpoint Analysis: /users (GET)                                     ┃ │
│ ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛ │
│                                                                                                                 │
│                                                                                                                 │
│                                         1. High-Level Goal and Use Case                                         │
│                                                                                                                 │
│ Goal: To retrieve a list of users from the system.                                                              │
│                                                                                                                 │
│ Use Case: This endpoint is designed for scenarios where an application needs to display a list of users. This   │
│ could be for administrative dashboards to manage users, for displaying user profiles within an application, or  │
│ for any other feature that requires accessing user data. The endpoint supports pagination and filtering by      │
│ role, making it flexible for various display and data retrieval needs.                                          │
│                                                                                                                 │
│                                                                                                                 │
│                                           2. API Endpoint Explanation                                           │
│                                                                                                                 │
│ The /users endpoint, when accessed via the GET HTTP method, allows clients to fetch a paginated list of users.  │
│                                                                                                                 │
│ Request-Response Cycle and Working Flow:                                                                        │
│                                                                                                                 │
│  1 Client Request: The client initiates a GET request to the /users path.                                       │
│     • Optional Query Parameters: The client can include the following query parameters to customize the         │
│       request:                                                                                                  │
│        • page (integer, optional): Specifies the desired page number for the results. Defaults to 1.            │
│        • limit (integer, optional): Determines the number of users to return per page. Must be between 1 and    │
│          100. Defaults to 20.                                                                                   │
│        • role (string, optional): Filters the user list to include only users with the specified role (admin,   │
│          user, or guest).                                                                                       │
│  2 Server Processing:                                                                                           │
│     • The server receives the request and parses the query parameters.                                          │
│     • It applies the specified page, limit, and role filters to query the user data store.                      │
│     • The server constructs a response containing the relevant user data.                                       │
│  3 Server Response:                                   